***

Preparing Workspace

***

In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format

In [2]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_main = os.path.join(path_sp, 'Data')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'BEA Data')
path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_data = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators','Seth', 'BEA', 'BEAbyIndustry.csv')

path_config0 = os.path.join(path_git, 'config')


In [3]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

Output_1

***

In [4]:
# def notin(element, collection):
#     return element not in collection

sacog = ["Yuba City", "Sacramento-Roseville-Folsom"]
mtc = ["San Francisco-Oakland-Berkeley", "Santa Rosa-Petaluma",
       "Vallejo", "Napa", "San Jose-Sunnyvale-Santa Clara"]
scag = ["Los Angeles-Long Beach-Anaheim",
        "Riverside-San Bernardino-Ontario",
        "Oxnard-Thousand Oaks-Ventura",
        "El Centro"]

df_rdp = pd.read_csv(path_data, skiprows=3, na_values="(D)")
display(df_rdp.head())

df_rdp.columns = df_rdp.columns.str.strip()


# Filtering and selecting columns
df_rdp = df_rdp[df_rdp['LineCode'] < 87]

# rdp[['GeoName', 'Level']]
# rdp = rdp[rdp['GeoName']]
#rdp = rdp.loc[:, ['GeoName', 'Level'] + [str(year) for year in range(2018, 2023)]]
df_rdp = df_rdp.dropna(subset=['GeoName'])
df_rdp = df_rdp[df_rdp['Description'] != 'Addenda:']

# Replacing parts of GeoName

df_rdp['GeoName'] = df_rdp['GeoName'].str.replace(r"\(.*", "", regex=True)
df_rdp['GeoName'] = df_rdp['GeoName'].str.replace(r",.*" , "", regex=True)

df_rdp['MPO'] = np.where(df_rdp['GeoName'].isin(sacog), 'SACOG'
              , np.where(df_rdp['GeoName'].isin(mtc  ), 'MTC'
              , np.where(df_rdp['GeoName'].isin(scag ), 'SCAG'
              , np.where(df_rdp['GeoName'] == "San Diego-Chula Vista-Carlsbad", "SANDAG"
              , df_rdp['GeoName'].str.replace('-', ' ').str.split().str[0])))
)

df_rdp['MPO'] = df_rdp['MPO'].str.replace(r"\(.*", "", regex=True)
df_rdp['MPO'] = df_rdp['MPO'].str.replace(r",.*" , "", regex=True)


df_rdp = df_rdp[["GeoName", "MPO", "Level", "Description", "2017", "2018", "2019", "2020", "2021", "2022"]]
df_rdp.columns = ["MSA", "MPO", "Level", "Industry", "2017", "2018", "2019", "2020", "2021", "2022"]

display(df_rdp.head())


df_rdp = pd.melt(df_rdp.reset_index(), id_vars=['MSA', 'MPO', 'Level', 'Industry'], value_vars=['2017', '2018', '2019', '2020', '2021', '2022'])

df_rdp.columns = ["MSA", "MPO", "Level", "Industry", "Year", "GRP"]
df_rdp = df_rdp[["Year", "MSA", "Industry", "GRP"]]
df_rdp = df_rdp[df_rdp['Industry'] == 'All industry total']
# df_rdp = df_rdp[~df_rdp["MPO"].isin(["MTC", "SANDAG", "SCAG", "SACOG"])]
df_rdp = df_rdp.sort_values(['MSA', 'Year'], ascending = [True, True]).reset_index(drop = True)


df_rdp['AnnualPctDiff'] = df_rdp['GRP'].pct_change()*100
df_rdp.loc[df_rdp['Year'] == '2017', 'AnnualPctDiff'] = np.nan
df_rdp.loc[df_rdp['AnnualPctDiff'] == np.inf, 'AnnualPctDiff'] = np.nan

year_max = np.max(df_rdp['Year'].unique())
year_min = np.min(df_rdp['Year'].unique())


df_rdp.head(10)


,GeoFips,GeoName,LineCode,Level,Description,2017,2018,2019,2020,2021,2022
0,12420,"Austin-Round Rock-Georgetown, TX (Metropolitan...",1.0,1.0,All industry total,141102903.0,151747891.0,164432513.0,171619050.0,195828575.0,222054436.0
1,12420,"Austin-Round Rock-Georgetown, TX (Metropolitan...",2.0,2.0,Private industries,124745562.0,134683231.0,146802709.0,152786094.0,176402729.0,201700555.0
2,12420,"Austin-Round Rock-Georgetown, TX (Metropolitan...",3.0,3.0,"Agriculture, forestry, fishing and hunting",31422.0,45787.0,30722.0,37701.0,37372.0,39393.0
3,12420,"Austin-Round Rock-Georgetown, TX (Metropolitan...",6.0,3.0,"Mining, quarrying, and oil and gas extraction",978382.0,1114801.0,1296493.0,1328372.0,984445.0,1045704.0
4,12420,"Austin-Round Rock-Georgetown, TX (Metropolitan...",10.0,3.0,Utilities,687391.0,NaN,NaN,NaN,983376.0,1178042.0


,MSA,MPO,Level,Industry,2017,2018,2019,2020,2021,2022
0,Austin-Round Rock-Georgetown,Austin,1.0,All industry total,141102903.0,151747891.0,164432513.0,171619050.0,195828575.0,222054436.0
1,Austin-Round Rock-Georgetown,Austin,2.0,Private industries,124745562.0,134683231.0,146802709.0,152786094.0,176402729.0,201700555.0
2,Austin-Round Rock-Georgetown,Austin,3.0,"Agriculture, forestry, fishing and hunting",31422.0,45787.0,30722.0,37701.0,37372.0,39393.0
3,Austin-Round Rock-Georgetown,Austin,3.0,"Mining, quarrying, and oil and gas extraction",978382.0,1114801.0,1296493.0,1328372.0,984445.0,1045704.0
4,Austin-Round Rock-Georgetown,Austin,3.0,Utilities,687391.0,NaN,NaN,NaN,983376.0,1178042.0


,Year,MSA,Industry,GRP,AnnualPctDiff
0,2017,Austin-Round Rock-Georgetown,All industry total,141102903.0,NaN
1,2018,Austin-Round Rock-Georgetown,All industry total,151747891.0,7.544131
2,2019,Austin-Round Rock-Georgetown,All industry total,164432513.0,8.359010
3,2020,Austin-Round Rock-Georgetown,All industry total,171619050.0,4.370509
4,2021,Austin-Round Rock-Georgetown,All industry total,195828575.0,14.106549
5,2022,Austin-Round Rock-Georgetown,All industry total,222054436.0,13.392254
6,2017,Charlotte-Concord-Gastonia,All industry total,168119917.0,NaN
7,2018,Charlotte-Concord-Gastonia,All industry total,175798075.0,4.567072
8,2019,Charlotte-Concord-Gastonia,All industry total,185714343.0,5.640715
9,2020,Charlotte-Concord-Gastonia,All industry total,190479926.0,2.566082


In [5]:

df_rdp2 = df_rdp.copy()
df_rdp2 = df_rdp2[(df_rdp2['Year'] == year_max) | (df_rdp2['Year'] == year_min)]
df_rdp2 = df_rdp2.drop('AnnualPctDiff', axis = 1)
df_rdp2['PctChange_2017_2022'] = df_rdp2['GRP'].pct_change()*100
df_rdp2.loc[df_rdp2['Year'] == year_min, 'PctChange_2017_2022'] = np.nan
df_rdp2.loc[df_rdp2['PctChange_2017_2022'] == np.inf, 'PctChange_2017_2022'] = np.nan
df_rdp2 = df_rdp2[df_rdp2['Year'] == year_max]
df_rdp2 = df_rdp2.drop(['Year', 'Industry', 'GRP'], axis = 1)

df_rdp3 = df_rdp.copy()
df_rdp3 = df_rdp3.groupby(['MSA'], as_index = False).agg(AnnualPctDiff_2017_2022 = ('AnnualPctDiff', 'mean'))

df_rdp2 = df_rdp2.merge(df_rdp3, on = 'MSA')

df_rdp   = df_rdp.sort_values(['MSA', 'Year'], ascending = [True, False])
df_rdp2 = df_rdp2.sort_values(['MSA'        ], ascending = [True       ])

df_rdp  = df_rdp .reset_index(drop = True)
df_rdp2 = df_rdp2.reset_index(drop = True)

display(df_rdp.head())
display(df_rdp2.head())


,Year,MSA,Industry,GRP,AnnualPctDiff
0,2022,Austin-Round Rock-Georgetown,All industry total,222054436.0,13.392254
1,2021,Austin-Round Rock-Georgetown,All industry total,195828575.0,14.106549
2,2020,Austin-Round Rock-Georgetown,All industry total,171619050.0,4.370509
3,2019,Austin-Round Rock-Georgetown,All industry total,164432513.0,8.359010
4,2018,Austin-Round Rock-Georgetown,All industry total,151747891.0,7.544131


,MSA,PctChange_2017_2022,AnnualPctDiff_2017_2022
0,Austin-Round Rock-Georgetown,57.370565,9.554491
1,Charlotte-Concord-Gastonia,36.160349,6.405127
2,Cincinnati,28.159099,5.150007
3,Cleveland-Elyria,26.541163,4.898521
4,Columbus,30.149460,5.461733


In [ ]:
sample_type = 'BEA'
indicator_name = 'Output_1'
year_start = 2017
year_end = 2022
geography = 'MSA'

# Create about documentation page for export
df_about = write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0)

print("Visual representation of the output for:", indicator_name)
display(df_about)

In [ ]:
# df_rdp .to_csv(os.path.join(path_agol, 'Output_1', 'Output_1_MSA_BEA.csv'      ), index = False)
# df_rdp2.to_csv(os.path.join(path_agol, 'Output_1', 'Output_1_MSA_BEA_rates.csv'), index = False)

In [ ]:
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Economy', 'Jobs', 'Output_1 GRP')

with pd.ExcelWriter(os.path.join(path_out, 'Output_1 MSA BEA.xlsx'), engine='openpyxl') as writer:
    df_about.to_excel(writer, index = False, sheet_name = 'About', header = False)
    df_rdp  .to_excel(writer, index = False, sheet_name = 'MSA'                  )
    df_rdp2 .to_excel(writer, index = False, sheet_name = 'Rates'                )

In [ ]:

path_out = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data"

with pd.ExcelWriter(os.path.join(path_out, 'Output_1 MSA BEA.xlsx'), engine='openpyxl') as writer:
    df_about.to_excel(writer, index = False, sheet_name = 'About', header = False)
    df_rdp  .to_excel(writer, index = False, sheet_name = 'MSA'                  )
    df_rdp2 .to_excel(writer, index = False, sheet_name = 'Rates'                )


print('')
print("Successfully exported!")